# Baseline 2 — Multi-task EfficientNet-B0

EfficientNet overview, Exp01–05 and historical full-data workflow. This notebook preserves the recorded experiment and reviews existing evidence. Read the local record guide before running any code.


## Local record guide — Exp01–05

**Objective:** EfficientNet overview, Exp01–05 and historical full-data workflow.

**Architecture:** Baseline 2 uses one shared EfficientNet-B0 backbone and separate 55-class crop and 175-class disease heads. The original method, dataset, training and analysis sections follow. The saved-evidence section at the end adds tables, existing figures and a concise discussion for this stage.

**Reading this completed record:** original execution counts, outputs and error records are preserved from `baseline2.ipynb`. They describe the historical execution, before these presentation/path changes. New Markdown is derived from explicitly linked artifacts; it is not a newly executed experiment.

The local setup cell locates the project root before Python imports. All original code cells are guarded by `RUN_RECORDED_WORKFLOW=False` so Run All in review mode does not train, infer, or rewrite reports. Keep that setting for review; read the saved results directly. Reproducing a historical workflow requires deliberate configuration of its controls and a fresh output directory. Original overwrite protections remain.

**Navigation:** introduction → setup → dataset → model/method → training/results → visualisation → discussion. Some optional historical cells were never executed; do not treat them as results.


## 1. Environment and configuration

The configuration below records the Exp04 development recipe. See the [README](../../README.md) for environment setup and local artifacts. The default training-image root is the sibling DPD_Small_dataset folder. Set the B2_IMAGE_ROOT environment variable to override it. Relative overrides resolve from the project root.

The saved reports can be reviewed directly. Executing the split checks requires the original local splits_v2 files. Run switches for training, diagnostics, synthetic checks and final evaluation are disabled by default.

In [ ]:
# Local review setup: no training, inference, or result writes.
from pathlib import Path
import os
import sys
_override = os.environ.get("B2_PROJECT_ROOT")
_start = Path(_override).expanduser().resolve() if _override else Path.cwd().resolve()
_candidates = [_start] if _override else [_start, *_start.parents]
PROJECT_DIR = next((p for p in _candidates if (p / "model.py").is_file()
                    and (p / "evaluate.py").is_file() and (p / "splits_v2").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open the Baseline2 workspace or set B2_PROJECT_ROOT to its root.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
RUN_RECORDED_WORKFLOW = False  # Keep False to review this completed record.
print("Project:", PROJECT_DIR, "| Historical workflow disabled; saved outputs retained.")


In [ ]:
if RUN_RECORDED_WORKFLOW:
    from pathlib import Path
    from dataclasses import dataclass
    import copy
    import gc
    import json
    import math
    import os
    import sys
    import platform
    import random
    import tempfile
    import time
    import warnings

    import numpy as np
    import pandas as pd
    from PIL import Image
    import matplotlib.pyplot as plt
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, WeightedRandomSampler
    import torchvision
    from torchvision import transforms as T
    from torchvision.transforms import InterpolationMode

    # Locate the project before importing its modules, including from nested notebook folders.
    kernel_dir = PROJECT_DIR.resolve()
    PROJECT_DIR = next(
        (folder for folder in (kernel_dir, *kernel_dir.parents)
         if all((folder / name).is_file() for name in ("model.py", "evaluate.py"))),
        None,
    )
    if PROJECT_DIR is None:
        raise FileNotFoundError("Start Jupyter in the Baseline2 folder or one of its subfolders.")
    sys.path.insert(0, str(PROJECT_DIR))

    from model import MultiTaskEfficientNetB0, load_checkpoint
    from evaluate import (DPDDataset, MEAN, STD, evaluate_loader, generalisation_metrics,
                          validate_development_splits, validation_transform,
                          make_candidates, require_candidate_coverage,
                          sha256_file, synthetic_self_test)

    SPLIT_DIR = PROJECT_DIR / "splits_v2"
    IMAGE_ROOT = Path(os.environ.get(
        "B2_IMAGE_ROOT", str(PROJECT_DIR.parent / "DPD_Small_dataset")
    )).expanduser()
    IMAGE_ROOT = (PROJECT_DIR / IMAGE_ROOT).resolve()
    OUTPUT_ROOT = PROJECT_DIR / "outputs"

    RUN_SMOKE_CHECKS = False
    RUN_STAGE1 = False       # Deliberately OFF: explicitly change only before starting a new run.
    RUN_DIAGNOSTICS = False  # Safe development-only analysis of an already-saved model.
    DIAG_EXPERIMENT_ID = "exp04"  # Historical error-analysis source (disabled by default).

    CONFIG = {
        "experiment_id": "exp04",
        "changed_parameter": "Tempered disease-balanced sampling (alpha=0.25) versus Exp01",
        "backbone": "efficientnet_b0", "pretrained_weights": "IMAGENET1K_V1",
        "crop_classes": 55, "disease_classes": 175, "image_size": 224,
        "batch_size": 32, "max_epochs": 30, "patience": 5, "min_delta": 1e-6,
        "backbone_lr": 1e-5, "head_lr": 1e-3, "dropout": 0.2,
        "weight_decay": 1e-4, "optimizer": "AdamW", "scheduler": "CosineAnnealingLR",
        "crop_loss_weight": 1.0, "disease_loss_weight": 1.0,
        "seed": 42, "num_workers": 2, "prefetch_factor": 2, "amp": True,
        "cudnn_benchmark": False,  # Keep deterministic for controlled comparison.
        "crop_scale": [0.85, 1.0], "flip_probability": 0.5,
        "rotation_degrees": 10, "brightness": 0.1, "contrast": 0.1,
        "interpolation": "bicubic", "mean": list(MEAN), "std": list(STD),
        "candidate_mode": "all_pairs", "candidate_catalog": None,
        "sampler_mode": "disease_tempered", "sampler_alpha": 0.25,
    }
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    assert (PROJECT_DIR / "model.py").is_file() and (PROJECT_DIR / "evaluate.py").is_file(), "Check PROJECT_DIR"
    print("PyTorch", torch.__version__, "Torchvision", torchvision.__version__)
    print("CUDA", torch.cuda.is_available(), "GPU", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
    print("Experiment", CONFIG["experiment_id"], "patience", CONFIG["patience"],
          "dropout", CONFIG["dropout"], "requested_workers", CONFIG["num_workers"])
    print("Stage 1 enabled:", RUN_STAGE1, "Diagnostics enabled:", RUN_DIAGNOSTICS)


## 2. Dataset integrity — development split

The original **8,000 training images** were partitioned during development into 6,905 train-fit, 695 seen validation, and 400 unseen-composition validation images. Split-validation code checks IDs, labels, counts, held-out combinations, and overlap. These are **development data**, not official test images. In final Stage 2B, all 8,000 images were recombined for training; the former validation images are therefore **not independent evaluation data** for the final checkpoint.


In [ ]:
if RUN_RECORDED_WORKFLOW:
    frames, split_report = validate_development_splits(SPLIT_DIR)
    print(json.dumps(split_report, indent=2))
    canonical_metadata = pd.concat(frames.values(), ignore_index=True)
    class_mappings = {
        task: {str(int(row[0])): str(row[1]) for row in
               canonical_metadata[[f"{task}_id", f"{task}_name"]].drop_duplicates().itertuples(index=False, name=None)}
        for task in ("crop", "disease", "composition")
    }
    composition_mapping = canonical_metadata[
        ["composition_id", "crop_id", "disease_id", "composition_name"]
    ].drop_duplicates().sort_values("composition_id").to_dict(orient="records")
    print("Original IDs retained; no new class numbering.")


## 3. Preprocessing and composition scoring

Training uses random resized crops to 224 pixels, horizontal flips, rotations, and brightness and contrast jitter. Evaluation uses bicubic resize to 256 pixels on the shorter side, a 224-pixel centre crop, and ImageNet normalisation. Candidate pairs are global and fixed for each image. Original B2 ranks them by log P(crop) + log P(disease).

In [ ]:
if RUN_RECORDED_WORKFLOW:
    def training_transform(config):
        return T.Compose([
            T.RandomResizedCrop(config["image_size"], scale=tuple(config["crop_scale"]),
                                interpolation=InterpolationMode.BICUBIC),
            T.RandomHorizontalFlip(p=config["flip_probability"]),
            T.RandomRotation(config["rotation_degrees"], interpolation=InterpolationMode.BICUBIC),
            T.ColorJitter(brightness=config["brightness"], contrast=config["contrast"]),
            T.ToTensor(), T.Normalize(MEAN, STD),
        ])

    candidates = make_candidates(CONFIG["candidate_mode"], CONFIG["candidate_catalog"])
    require_candidate_coverage(candidates, list(frames.values()))
    print(f"Candidate mode: {CONFIG['candidate_mode']}; {len(candidates):,} global candidate pairs")
    if CONFIG["candidate_mode"] == "all_pairs":
        print("PROVISIONAL: confirm the same candidate policy with Baseline 1 before comparing results.")


## 4. Reproducibility and data loaders

Sets random seeds and defines worker/startup helpers. Multi-worker loading is probed when training is explicitly enabled; worker count is a performance setting, **not** a change to the model architecture. Windows notebook processes may require `num_workers=0` if worker startup fails.


In [ ]:
if RUN_RECORDED_WORKFLOW:
    def seed_everything(seed, config=None):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        fast = bool((config or {}).get("cudnn_benchmark", False))
        torch.backends.cudnn.benchmark = fast
        torch.backends.cudnn.deterministic = not fast



    def make_loader(dataset, config, shuffle=False, workers=None,
                    persistent=True, sampler=None):

        n = int(config["num_workers"] if workers is None else workers)

        kw = dict(
            batch_size=int(config["batch_size"]),
            num_workers=n,
            pin_memory=DEVICE.type == "cuda",
            generator=torch.Generator().manual_seed(int(config["seed"])),
            drop_last=False,
        )

        if sampler is not None:
            if shuffle:
                raise ValueError("Cannot use shuffle=True with sampler")
            kw["sampler"] = sampler
        else:
            kw["shuffle"] = shuffle

        if n:
            kw.update(
                persistent_workers=persistent,
                prefetch_factor=int(config.get("prefetch_factor", 2)),
                timeout=30,
            )

        return DataLoader(dataset, **kw)



    def preflight_workers(datasets, config):
        n = int(config["num_workers"])
        if n == 0:
            print("Worker preflight: using 0 (single-process loader)")
            return 0
        print(f"Worker preflight: trying num_workers={n} (no training)...")
        try:
            # Temporary loaders are discarded, so the real run gets a fresh seed/order.
            for name in ("train_fit", "val_seen", "val_unseen"):
                probe = make_loader(datasets[name], config, shuffle=False, workers=n, persistent=False)
                it = iter(probe)
                batch = next(it)
                assert batch["image"].shape[1:] == (3, 224, 224)
                del it, probe
            print(f"Workers initialized: num_workers={n}")
            return n
        except Exception as exc:
            warnings.warn(f"Worker start failed: {type(exc).__name__}: {exc}. Falling back to num_workers=0.")
            return 0


    @dataclass
    class EarlyStopping:
        patience: int = 15
        min_delta: float = 1e-6
        best: float = -math.inf
        bad_epochs: int = 0

        def update(self, value):
            if not math.isfinite(value):
                raise ValueError("Nonfinite validation harmonic mean")
            improved = value > self.best + self.min_delta
            if improved:
                self.best = value
                self.bad_epochs = 0
            else:
                self.bad_epochs += 1
            return improved, self.bad_epochs >= self.patience


    def atomic_checkpoint(payload, path):
        path = Path(path)
        temporary = path.with_suffix(".tmp")
        torch.save(payload, temporary)
        temporary.replace(path)


    def environment_details():
        return {"python": platform.python_version(), "torch": str(torch.__version__),
                "torchvision": str(torchvision.__version__), "numpy": np.__version__,
                "pandas": pd.__version__, "device": str(DEVICE),
                "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
                "cuda_runtime": torch.version.cuda}


## 5. Synthetic implementation checks

The optional checks use synthetic tensors to verify shapes, gradients, metrics, checkpoint restoration and early stopping. They are disabled by default.

In [ ]:
if RUN_RECORDED_WORKFLOW:
    def run_smoke_checks():
        synthetic_self_test()
        seed_everything(CONFIG["seed"])
        smoke = MultiTaskEfficientNetB0(pretrained=False).to(DEVICE)
        assert len(smoke.backbone) == 3
        assert smoke.crop_head[1].in_features == smoke.disease_head[1].in_features == 1280
        assert not ({id(p) for p in smoke.crop_head.parameters()} &
                    {id(p) for p in smoke.disease_head.parameters()})
        groups = smoke.parameter_groups()
        group_ids = [id(p) for group in groups for p in list(group["params"])]
        assert len(group_ids) == len(set(group_ids)) == len(list(smoke.parameters()))
        # Recreate groups because the generator above was consumed.
        optimizer = torch.optim.AdamW(smoke.parameter_groups(), weight_decay=1e-4)
        scaler = torch.amp.GradScaler("cuda", init_scale=128.0, enabled=DEVICE.type == "cuda")
        images = torch.randn(2, 3, 224, 224, device=DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            output = smoke(images)
            assert output["crop_logits"].shape == (2, 55)
            assert output["disease_logits"].shape == (2, 175)
            lc = nn.functional.cross_entropy(output["crop_logits"], torch.tensor([0, 54], device=DEVICE))
            ld = nn.functional.cross_entropy(output["disease_logits"], torch.tensor([0, 174], device=DEVICE))
        shared_parameter = next(smoke.backbone.parameters())
        for loss in (lc, ld):
            gradient = torch.autograd.grad(loss, shared_parameter, retain_graph=True)[0]
            assert torch.isfinite(gradient).all() and gradient.abs().sum() > 0
        loss = lc + ld
        scaler.scale(loss).backward()

        # Unscale gradients before checking them
        scaler.unscale_(optimizer)

        for module in (smoke.backbone, smoke.crop_head, smoke.disease_head):
            grads = [p.grad for p in module.parameters() if p.grad is not None]
            assert grads and all(torch.isfinite(g).all() for g in grads)

        scaler.step(optimizer)
        scaler.update()
        smoke.eval() # Switch to evaluation mode before comparing predictions
        with torch.no_grad():
            before = smoke(images)["crop_logits"].cpu()
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "synthetic.pt"
            atomic_checkpoint({"model_state": smoke.state_dict(), "config": CONFIG}, path)
            restored, _ = load_checkpoint(path, DEVICE)
            with torch.no_grad():
                after = restored(images)["crop_logits"].cpu()
            assert torch.isfinite(before).all()
            assert torch.isfinite(after).all()

            print("Maximum prediction difference:",
                (before - after).abs().max().item())

            torch.testing.assert_close(
                before,
                after,
                rtol=1e-4,
                atol=1e-5
            )
        stopper = EarlyStopping(patience=3)
        assert stopper.update(0) == (True, False)
        for _ in range(2):
            assert stopper.update(0) == (False, False)
        assert stopper.update(0) == (False, True)
        assert stopper.update(0.1) == (True, False)
        assert stopper.update(0.1 + 1e-8) == (False, False)
        del smoke, restored, images, output, loss, lc, ld, optimizer
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        print("Architecture, gradients, optimizer, checkpoint, early stopping and metrics passed.")

    if RUN_SMOKE_CHECKS:
        run_smoke_checks()


## 6. Stage 1 — training and validation

Both heads and the backbone are trainable. The objective sums equally weighted crop and disease cross-entropies. AdamW uses separate backbone and head learning rates, followed by cosine annealing. Checkpoint selection and early stopping use the harmonic mean of seen and unseen composition Top-1. This metric penalises an imbalance between the two accuracies.

In [ ]:
if RUN_RECORDED_WORKFLOW:
    def train_one_epoch(model, loader, optimizer, scaler, device, amp=True):
        model.train()
        totals = np.zeros(3, dtype=float)
        n_images = 0
        correct_crop = correct_disease = correct_pair = 0
        criterion = nn.CrossEntropyLoss()
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            crop = batch["crop"].to(device, non_blocking=True)
            disease = batch["disease"].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            try:
                with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
                    result = model(images)
                    crop_loss = criterion(result["crop_logits"], crop)
                    disease_loss = criterion(result["disease_logits"], disease)
                    total_loss = crop_loss + disease_loss
                if not torch.isfinite(total_loss):
                    raise RuntimeError("Nonfinite training loss; stop and inspect the inputs/configuration.")
                scaler.scale(total_loss).backward()
                scaler.step(optimizer)
                scaler.update()
            except torch.cuda.OutOfMemoryError as error:
                raise RuntimeError("CUDA out of memory. Restart the kernel, explicitly set batch_size=16, "
                                   "record the change, and rerun as a new experiment. No setting was changed automatically.") from error
            count = images.shape[0]
            pc = result["crop_logits"].argmax(1)
            pd_ = result["disease_logits"].argmax(1)
            correct_crop += int((pc == crop).sum())
            correct_disease += int((pd_ == disease).sum())
            correct_pair += int(((pc == crop) & (pd_ == disease)).sum())
            totals += count * np.array([total_loss.item(), crop_loss.item(), disease_loss.item()])
            n_images += count
        if not n_images:
            raise ValueError("Empty training loader")
        result_metrics = dict(zip(("total_loss", "crop_loss", "disease_loss"), (totals / n_images).tolist()))
        result_metrics.update(crop_top1=correct_crop / n_images,
                              disease_top1=correct_disease / n_images,
                              composition_top1=correct_pair / n_images)
        return result_metrics


    def plot_training_history(history, best_epoch, path):
        table = pd.DataFrame(history)
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        axes[0, 0].plot(table.epoch, table.train_total_loss, label="Training")
        axes[0, 0].plot(table.epoch, table.seen_total_loss, label="Seen validation")
        axes[0, 0].plot(table.epoch, table.unseen_total_loss, label="Unseen validation")
        axes[0, 0].set(title="Total cross-entropy loss", ylabel="Mean loss")
        axes[0, 1].plot(table.epoch, table.train_crop_loss, label="Crop")
        axes[0, 1].plot(table.epoch, table.train_disease_loss, label="Disease")
        axes[0, 1].set(title="Training task losses", ylabel="Mean loss")
        axes[1, 0].plot(table.epoch, table.seen_composition_top1 * 100, label="Seen-like")
        axes[1, 0].plot(table.epoch, table.unseen_composition_top1 * 100, label="Unseen-like")
        axes[1, 0].set(title="Composition Top-1 validation", ylabel="Accuracy (%)")
        axes[1, 1].plot(table.epoch, table.composition_harmonic_mean * 100, label="Harmonic mean")
        axes[1, 1].set(title="Development selection criterion", ylabel="Accuracy (%)")
        for ax in axes.flat:
            ax.axvline(best_epoch, linestyle="--", color="black", alpha=.5, label=f"Best epoch {best_epoch}")
            ax.set_xlabel("Epoch")
            ax.grid(alpha=.2)
            ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(path, dpi=160)
        return fig


    def plot_task_accuracy(history, best_epoch, path):
        table = pd.DataFrame(history)
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        for ax, task in zip(axes, ("crop", "disease", "composition")):
            for prefix, label in (("train", "Train"), ("seen", "Seen val"), ("unseen", "Unseen val")):
                col = f"{prefix}_{task}_top1"
                ax.plot(table.epoch, table[col] * 100, label=label)
            ax.axvline(best_epoch, linestyle="--", color="black", alpha=.5)
            ax.set(title=f"{task.title()} Top-1", xlabel="Epoch", ylabel="Accuracy (%)")
            ax.grid(alpha=.2)
            ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(path, dpi=160)
        plt.show()
        return fig


    def fit_stage1(config, train_loader, seen_loader, unseen_loader, candidate_pairs,
                   metadata_report, mappings, composition_map, output_root=OUTPUT_ROOT):
        seed_everything(config["seed"], config)
        run_config = copy.deepcopy(config)
        run_config["candidate_catalog"] = (str(config["candidate_catalog"])
                                             if config["candidate_catalog"] is not None else None)
        # A failed pretrained download is an error, never a random-weight fallback.
        model = MultiTaskEfficientNetB0(dropout=config["dropout"], pretrained=True).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameter_groups(config["backbone_lr"], config["head_lr"]),
                                     weight_decay=config["weight_decay"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])
        amp_enabled = config["amp"] and DEVICE.type == "cuda"
        scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
        stopper = EarlyStopping(config["patience"], config["min_delta"])
        output_dir = Path(output_root) / config["experiment_id"]
        output_dir.mkdir(parents=True, exist_ok=False)
        environment = environment_details()
        (output_dir / "config.json").write_text(json.dumps({"config": run_config, "environment": environment,
             "development_split": metadata_report, "class_mappings": mappings,
             "composition_mapping": composition_map,
             "candidate_pairs": candidate_pairs.cpu().tolist(),
             "candidate_policy_status": "provisional" if config["candidate_mode"] == "all_pairs" else "user-supplied group catalogue",
             "accuracy_units": "fractions in [0,1]"}, indent=2), encoding="utf-8")
        history = []
        best_epoch = None
        start = time.perf_counter()
        for epoch in range(1, config["max_epochs"] + 1):
            epoch_start = time.perf_counter()
            rates = [group["lr"] for group in optimizer.param_groups]
            training = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE, amp_enabled)
            train_elapsed = time.perf_counter() - epoch_start
            seen, _ = evaluate_loader(model, seen_loader, DEVICE, candidate_pairs, amp_enabled)
            seen_elapsed = time.perf_counter() - epoch_start - train_elapsed
            unseen, _ = evaluate_loader(model, unseen_loader, DEVICE, candidate_pairs, amp_enabled)
            unseen_elapsed = time.perf_counter() - epoch_start - train_elapsed - seen_elapsed
            joint = generalisation_metrics(seen, unseen)
            row = {"epoch": epoch, "backbone_lr": rates[0], "head_lr": rates[1],
                   **{f"train_{key}": value for key, value in training.items()},
                   **{f"seen_{key}": value for key, value in seen.items()},
                   **{f"unseen_{key}": value for key, value in unseen.items()},
                   "composition_harmonic_mean": joint["composition_harmonic_mean"],
                   "seen_unseen_gap": joint["seen_unseen_gap"],
                   "epoch_seconds": time.perf_counter() - epoch_start,
                   "train_seconds": train_elapsed, "seen_eval_seconds": seen_elapsed,
                   "unseen_eval_seconds": unseen_elapsed}
            history.append(row)
            pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)
            improved, stop = stopper.update(joint["composition_harmonic_mean"])
            scheduler.step()
            if improved:
                best_epoch = epoch
                best_metrics = {"epoch": epoch, "seen": seen, "unseen": unseen, **joint}
                atomic_checkpoint({"stage": 1, "epoch": epoch, "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
                    "scaler_state": scaler.state_dict(), "config": run_config,
                    "candidate_pairs": candidate_pairs.cpu().tolist(),
                    "class_mappings": mappings, "composition_mapping": composition_map,
                    "development_split": metadata_report, "environment": environment,
                    "best_metrics": best_metrics, "parameter_count": sum(p.numel() for p in model.parameters())},
                    output_dir / "best_development.pt")
            print(f"Epoch {epoch:02d} | loss {training['total_loss']:.4f} | "
                  f"seen {seen['composition_top1']:.2%} | unseen {unseen['composition_top1']:.2%} | "
                  f"HM {joint['composition_harmonic_mean']:.2%} | {row['epoch_seconds']:.1f}s" + (" * best" if improved else ""))
            if stop:
                print(f"Early stopping after {config['patience']} consecutive non-improving epochs.")
                break
        best = torch.load(output_dir / "best_development.pt", map_location=DEVICE, weights_only=True)
        model.load_state_dict(best["model_state"])
        model.eval()
        validation_metrics = {**best["best_metrics"], "epochs_run": len(history),
                              "training_seconds": time.perf_counter() - start,
                              "accuracy_units": "fractions in [0,1]"}
        (output_dir / "validation_metrics.json").write_text(json.dumps(validation_metrics, indent=2), encoding="utf-8")
        log = {"experiment_id": config["experiment_id"], "changed_parameter": config["changed_parameter"],
               "best_epoch": best_epoch, **generalisation_metrics(best["best_metrics"]["seen"], best["best_metrics"]["unseen"]),
               "candidate_mode": config["candidate_mode"],
               **{f"best_seen_{k}": v for k, v in best["best_metrics"]["seen"].items() if k.endswith("top1") or k.endswith("top5")},
               **{f"best_unseen_{k}": v for k, v in best["best_metrics"]["unseen"].items() if k.endswith("top1") or k.endswith("top5")},
               "checkpoint_path": str(output_dir / "best_development.pt"),
               "configuration": json.dumps(run_config, sort_keys=True)}
        pd.DataFrame([log]).to_csv(output_dir / "experiment_summary.csv", index=False)
        figure = plot_training_history(history, best_epoch, output_dir / "training_curves.png")
        plt.show()
        plot_task_accuracy(history, best_epoch, output_dir / "task_accuracy_curves.png")
        return model, history, validation_metrics, output_dir


## 7. Stage 1 execution control — completed; OFF

The experiment run switch is deliberately `False`. Historical Stage 1 experiments **Exp01–Exp05** are complete and should not be rerun or overwritten. Results and checkpoints remain in the owner's original local `outputs/` directory. This cell is preserved for reproducibility only.


In [ ]:
if RUN_RECORDED_WORKFLOW:
    if RUN_STAGE1:
        exp_dir = OUTPUT_ROOT / CONFIG["experiment_id"]
        if exp_dir.exists():
            raise FileExistsError(f"{exp_dir} exists. Use a NEW experiment ID; do not overwrite it.")
        frames, verified_report = validate_development_splits(SPLIT_DIR)
        candidates = make_candidates(CONFIG["candidate_mode"], CONFIG["candidate_catalog"])
        require_candidate_coverage(candidates, list(frames.values()))
        datasets = {
            name: DPDDataset(frame, IMAGE_ROOT,
                             training_transform(CONFIG) if name == "train_fit" else validation_transform())
            for name, frame in frames.items()
        }
        for dataset in datasets.values():
            for path in dataset.paths:
                with Image.open(path) as image:
                    image.convert("RGB").load()
        verified_report["image_paths_checked"] = True
        verified_report["image_decoding_checked"] = True

        actual_workers = preflight_workers(datasets, CONFIG)
        run_config = copy.deepcopy(CONFIG)
        run_config["requested_num_workers"] = int(CONFIG["num_workers"])
        run_config["num_workers"] = int(actual_workers)  # record EFFECTIVE workers in checkpoint
        if actual_workers != CONFIG["num_workers"]:
            run_config["data_loading_note"] = "Worker preflight failed; num_workers=0 fallback"
        print("Training will use num_workers:", actual_workers)
    
        # Exp04: Tempered Disease-Balanced Sampling
        train_sampler = None

        if run_config.get("sampler_mode") == "disease_tempered":
            disease_ids = frames["train_fit"]["disease_id"].to_numpy(
                dtype=np.int64
            )

            assert len(disease_ids) == len(datasets["train_fit"])

            counts = np.bincount(
                disease_ids,
                minlength=int(run_config["disease_classes"])
            )

            alpha = float(run_config["sampler_alpha"])

            weights = torch.as_tensor(
                1.0 / np.power(counts[disease_ids], alpha),
                dtype=torch.double
            )

            assert torch.isfinite(weights).all()
            assert (weights > 0).all()

            train_sampler = WeightedRandomSampler(
                weights=weights,
                num_samples=len(datasets["train_fit"]),
                replacement=True,
                generator=torch.Generator().manual_seed(
                    int(run_config["seed"])
                ),
            )

            print(
                f"Disease-balanced sampling enabled | "
                f"alpha={alpha} | "
                f"samples/epoch={len(datasets['train_fit'])}"
            )

        loaders = {
            name: make_loader(
                dataset,
                run_config,
                shuffle=(name == "train_fit" and train_sampler is None),
                sampler=(train_sampler if name == "train_fit" else None),
            )
            for name, dataset in datasets.items()
        }

        seed_everything(run_config["seed"], run_config)
        model, history, metrics, experiment_dir = fit_stage1(
            run_config, loaders["train_fit"], loaders["val_seen"], loaders["val_unseen"],
            candidates, verified_report, class_mappings, composition_mapping)
        print("Best development epoch:", metrics["epoch"])
        print("Saved to:", experiment_dir)
    else:
        print("Stage 1 OFF. Set RUN_STAGE1=True in Section 1 only after approving the run.")


## 8. Development error-analysis functions

Functions for development metrics, confusion matrices, macro-average precision/recall/F1, and image-level error inspection. Historical **Exp04 error analysis is already complete**. `RUN_DIAGNOSTICS=False` in Section 1 prevents additional execution. The code does not access the official-test split.


In [ ]:
if RUN_RECORDED_WORKFLOW:
    def metrics_and_confusion(records, truth_col, pred_col, class_count, dest, label):
        truth = np.array([int(r[truth_col]) for r in records])
        pred = np.array([int(r[pred_col]) for r in records])
        matrix = np.zeros((class_count, class_count), dtype=np.int64)
        np.add.at(matrix, (truth, pred), 1)
        pd.DataFrame(matrix, index=range(class_count), columns=range(class_count)).to_csv(dest / f"{label}_confusion.csv", index_label="true_id")
        tp = np.diag(matrix)
        support = matrix.sum(axis=1)
        predicted_count = matrix.sum(axis=0)
        precision = np.divide(tp, predicted_count, out=np.zeros(class_count), where=predicted_count > 0)
        recall = np.divide(tp, support, out=np.zeros(class_count), where=support > 0)
        f1 = np.divide(2 * precision * recall, precision + recall,
                       out=np.zeros(class_count), where=(precision + recall) > 0)
        present = support > 0   # exclude absent true classes from this split's macro average
        stats = {"macro_precision": float(precision[present].mean()),
                 "macro_recall": float(recall[present].mean()),
                 "macro_f1": float(f1[present].mean()),
                 "n_true_classes": int(present.sum())}
        pd.DataFrame({"class_id": range(class_count), "support": support,
                      "precision": precision, "recall": recall, "f1": f1}).to_csv(dest / f"{label}_per_class.csv", index=False)
        return stats, matrix


    def run_development_diagnostics(experiment_id=DIAG_EXPERIMENT_ID, example_images=6):
        from matplotlib import pyplot as plt
        folder = OUTPUT_ROOT / experiment_id
        checkpoint_path = folder / "best_development.pt"
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f"No Stage 1 checkpoint: {checkpoint_path}")
        model, ck = load_checkpoint(checkpoint_path, DEVICE)
        assert ck.get("stage") == 1, "Use Stage 1 development checkpoint only"
        split_frames, report = validate_development_splits(SPLIT_DIR)
        if report["sha256"] != ck["development_split"]["sha256"]:
            raise ValueError("Split files changed since checkpoint: refusing to evaluate")
        saved_candidates = torch.as_tensor(ck["candidate_pairs"], dtype=torch.long)
        dest = folder / "development_diagnostics"
        dest.mkdir(parents=True, exist_ok=True)
        mapping = ck["class_mappings"]
        reports = {}
        for split in ("val_seen", "val_unseen"):
            dataset = DPDDataset(split_frames[split], IMAGE_ROOT, validation_transform())
            loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0,
                                pin_memory=DEVICE.type == "cuda")
            scores, records = evaluate_loader(model, loader, DEVICE, saved_candidates,
                                              amp=DEVICE.type == "cuda", collect_predictions=True)
            assert len(records) == len(dataset)
            table = pd.DataFrame(records)
            table["crop_correct"] = table.crop_id == table.predicted_crop_id
            table["disease_correct"] = table.disease_id == table.predicted_disease_id
            table["composition_correct"] = ((table.crop_id == table.predicted_pair_crop_id) &
                                              (table.disease_id == table.predicted_pair_disease_id))
            table["error_type"] = np.select(
                [~table.crop_correct & ~table.disease_correct,
                 ~table.crop_correct, ~table.disease_correct],
                ["both_wrong", "crop_wrong", "disease_wrong"], default="correct")
            for task in ("crop", "disease"):
                for prefix in ("", "predicted_"):
                    name_col = prefix + task + "_name"
                    id_col = prefix + task + "_id"
                    table[name_col] = table[id_col].map(lambda x: mapping[task].get(str(int(x)), f"id_{x}"))
            # Only display/scalar columns. Ranked lists are available in evaluate_loader but omitted from CSV.
            keep = ["filename", "composition_id", "crop_id", "crop_name", "disease_id", "disease_name",
                    "predicted_crop_id", "predicted_crop_name", "predicted_disease_id", "predicted_disease_name",
                    "crop_correct", "disease_correct", "composition_correct", "error_type"]
            table[keep].to_csv(dest / f"{split}_predictions.csv", index=False)
            macro = {}
            for task, ncls in (("crop", 55), ("disease", 175)):
                m, _ = metrics_and_confusion(records, f"{task}_id", f"predicted_{task}_id", ncls,
                                             dest, f"{split}_{task}")
                macro[task] = m
            percomp = (table.groupby("composition_id")
                       .agg(images=("filename", "count"), crop_top1=("crop_correct", "mean"),
                            disease_top1=("disease_correct", "mean"),
                            composition_top1=("composition_correct", "mean"))
                       .sort_index())
            percomp.to_csv(dest / f"{split}_per_composition.csv")
            wrong = table[~table.composition_correct]
            errors = (wrong.groupby(["crop_name", "disease_name", "predicted_crop_name", "predicted_disease_name"])
                      .size().sort_values(ascending=False).head(10))
            print(f"\n{split}: Crop {scores['crop_top1']:.2%}, Disease {scores['disease_top1']:.2%}, "
                  f"Composition {scores['composition_top1']:.2%}")
            print("Macro P/R/F1 (supported labels):", {k: {m: round(v, 3) if isinstance(v, float) else v
                                                    for m, v in s.items()} for k, s in macro.items()})
            print("Top 10 misclassifications: ")
            print(errors.to_string() if len(errors) else "None")
            if split == "val_unseen":
                print("Unseen per-composition accuracy:")
                print(percomp.round(3).to_string())
                if example_images and len(wrong):
                    picks = wrong.head(example_images)
                    fig, axes = plt.subplots(1, len(picks), figsize=(3.0 * len(picks), 3.8))
                    axes = np.array([axes]).ravel() if len(picks) == 1 else np.array(axes).ravel()
                    for ax, (_, r) in zip(axes, picks.iterrows()):
                        with Image.open(IMAGE_ROOT / r.filename) as image:
                            ax.imshow(image.convert("RGB"))
                        ax.set_title(f"True: {r.crop_name} / {r.disease_name}\nPred: {r.predicted_crop_name} / {r.predicted_disease_name}", fontsize=8)
                        ax.axis("off")
                    fig.tight_layout()
                    fig.savefig(dest / "unseen_misclassified_examples.png", dpi=140, bbox_inches="tight")
                    plt.show()
            reports[split] = {"accuracy": scores, "macro": macro, "n_errors": int(len(wrong))}
        (dest / "diagnostic_summary.json").write_text(json.dumps(reports, indent=2), encoding="utf-8")
        print("Development diagnostics saved to:", dest)
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        return reports

    if RUN_DIAGNOSTICS:
        development_diagnostics = run_development_diagnostics()
    else:
        print("Development diagnostics OFF; set RUN_DIAGNOSTICS=True if you want to evaluate an EXISTING checkpoint.")


## 9. Development experiment selection

| Method | Seen Top-1 | Unseen Top-1 | Harmonic mean |
|---|---:|---:|---:|
| Exp04, original scoring | 66.47% | 2.25% | 4.35% |
| Exp06, calibration on Exp04 (γ = 4) | 46.19% | 6.50% | 11.40% |
| Exp07, two-view TTA on Exp04 | 66.47% | 2.25% | 4.35% |

Exp04 used tempered disease-balanced sampling (α = 0.25), which weights each image by its disease class frequency raised to −0.25. Its selected checkpoint was epoch 13. Exp06 selected the penalty on development validation, so its development score is subject to selection bias. Exp07 did not improve harmonic mean over Exp04 and was omitted from official evaluation.

Saved evidence is available for [Exp04](../../reports/development/exp04/validation_metrics.json), [Exp06](../../reports/development/exp06/calibration_summary.json), and [Exp07](../../reports/development/exp07/tta_summary.json).

## 10. Stage 2 — development fine-tuning and final fit (completed; OFF)

The following code is retained for reproducibility; **do not rerun existing fine-tuning or the full-data fit**. Both Stage 2 controls remain disabled.

- **10A — Development fine-tuning:** FT01–FT04 independently examined training choices against the unchanged development split, without touching the official test set.
- **10B — Final full-data training:** Exp04 was selected as parent checkpoint and fine-tuned using all **8,000 original training images** for **3 additional epochs**. Backbone LR `1e-6`; both heads LR `1e-4`; AdamW; tempered disease-balanced sampling (`alpha=0.25`). Final model was saved as `outputs/stage2_final/b2_final_exp04/final_full_data.pt`.
- The former validation images are part of the final training set. **No development validation or official testing was used to choose the final full-data epoch.**

Saved outputs and checkpoint are authoritative; do not recreate them merely to render this notebook.


In [ ]:
if RUN_RECORDED_WORKFLOW:
    # SECTION 10 — shared helpers. Definition only: safe to execute, NO training.
    STAGE2_DEVELOPMENT_PRESETS = {
        "ft01": {"crop_loss_weight": 1.0, "disease_loss_weight": 1.0,
                 "fine_tune_mode": "full"},
        "ft02": {"crop_loss_weight": 1.0, "disease_loss_weight": 1.5,
                 "fine_tune_mode": "full"},
        # Controlled partial fine-tuning versus FT01: ONLY trainability changes.
        "ft03": {"crop_loss_weight": 1.0, "disease_loss_weight": 1.0,
                 "fine_tune_mode": "partial_last2"},
        # cRT-inspired: Exp04 fixed representation, freshly initialized classifiers.
        "ft04": {"crop_loss_weight": 1.0, "disease_loss_weight": 1.0,
                 "fine_tune_mode": "head_only_retrain"},
    }


    def stage2_read_parent(checkpoint_path, allowed_stages=(1, "stage2_development")):
        """Load ONLY a saved training checkpoint; verify the unchanged development split."""
        checkpoint_path = Path(checkpoint_path)
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
        ck = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        if ck.get("stage") not in allowed_stages:
            raise ValueError(f"Invalid parent stage: {ck.get('stage')!r}")
        frames_local, report_local = validate_development_splits(SPLIT_DIR)
        if report_local["sha256"] != ck["development_split"]["sha256"]:
            raise ValueError("Development split files differ from checkpoint; stopping.")
        if tuple(len(frames_local[k]) for k in ("train_fit", "val_seen", "val_unseen")) != (6905, 695, 400):
            raise ValueError("Expected 6,905/695/400 development split sizes.")
        candidate_pairs_local = torch.as_tensor(ck["candidate_pairs"], dtype=torch.long)
        require_candidate_coverage(candidate_pairs_local, list(frames_local.values()))
        return checkpoint_path, ck, frames_local, report_local, candidate_pairs_local


    def stage2_build_sampler(frame, config):
        """Training-only weighted sampling; validation is never sampled."""
        mode = config.get("sampler_mode", "none")
        if mode in (None, "none"):
            return None
        if mode != "disease_tempered":
            raise ValueError(f"Unsupported sampler_mode: {mode}")
        alpha = float(config.get("sampler_alpha", 0.25))
        if not (0.0 <= alpha <= 1.0):
            raise ValueError("sampler_alpha must lie in [0,1]")
        ids = frame["disease_id"].to_numpy(dtype=np.int64)
        classes = int(config["disease_classes"])
        if len(ids) != len(frame) or len(ids) == 0 or ids.min() < 0 or ids.max() >= classes:
            raise ValueError("Invalid disease labels in training partition")
        counts = np.bincount(ids, minlength=classes)
        weights = torch.as_tensor(counts[ids].astype(np.float64) ** (-alpha), dtype=torch.double)
        if not bool(torch.isfinite(weights).all()) or not bool((weights > 0).all()):
            raise ValueError("Invalid training sampling weights")
        print(f"Sampler: {mode} | alpha={alpha} | replacement=True | draws/epoch={len(ids)}")
        return WeightedRandomSampler(weights, num_samples=len(ids), replacement=True,
                                     generator=torch.Generator().manual_seed(int(config["seed"])))


    def stage2_choose_workers(datasets, config, requested=4):
        """Try more workers; gracefully fall back to a known lower setting."""
        if int(requested) < 0:
            raise ValueError("num_workers must be >= 0")
        choices = list(dict.fromkeys([int(requested), 2, 0]))
        choices = [w for w in choices if w <= int(requested)] if requested < 2 else choices
        for n in choices:
            probe_config = dict(config, num_workers=n)
            actual = preflight_workers(datasets, probe_config)
            if actual == n:
                print(f"Stage 2 training will use num_workers={n}")
                return n
        raise RuntimeError("DataLoader preflight did not find a usable worker setting")


    def stage2_restore_model(checkpoint, device):
        """Restore actual trained weights, NEVER re-download or reinitialize ImageNet."""
        model = MultiTaskEfficientNetB0(dropout=float(checkpoint["config"]["dropout"]),
                                         pretrained=False).to(device)
        model.load_state_dict(checkpoint["model_state"], strict=True)
        if not all(p.requires_grad for p in model.parameters()):
            raise RuntimeError("Expected full-backbone + both-head fine-tuning")
        return model


    def stage2_configure_trainability(model, mode):
        """For FT03, freeze EfficientNet-B0 features[0:7] and train features[7:9] plus heads.

        The model remains architecturally identical to Exp04. Call this AFTER restoring
        the saved weights and BEFORE constructing the optimizer.
        """
        if mode == "full":
            if not all(p.requires_grad for p in model.parameters()):
                raise RuntimeError("Full fine-tuning requires all parameters trainable")
            return {"mode": mode, "frozen_features": [], "trainable_features": list(range(9))}
        if mode == "head_only_retrain":
            # All backbone blocks stay fixed, including BatchNorm buffers during training.
            for parameter in model.backbone.parameters():
                parameter.requires_grad_(False)
            for head in (model.crop_head, model.disease_head):
                for parameter in head.parameters():
                    parameter.requires_grad_(True)
            n_frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
            n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            if not n_frozen or not n_trainable:
                raise RuntimeError("FT04 requires a frozen backbone and trainable heads")
            print(f"FT04 Head-only Retraining: frozen backbone={n_frozen:,}, "
                  f"trainable crop/disease heads={n_trainable:,}")
            return {"mode": mode, "frozen_features": list(range(9)),
                    "trainable_features": [], "frozen_parameters": n_frozen,
                    "trainable_parameters": n_trainable, "frozen_bn_eval": True}
        if mode != "partial_last2":
            raise ValueError(f"Unknown fine-tune mode: {mode}")
        if not isinstance(model.backbone, nn.Sequential) or len(model.backbone) != 3:
            raise RuntimeError("Unexpected B2 backbone structure; inspect model.py before freezing")
        features = model.backbone[0]
        if not isinstance(features, nn.Sequential) or len(features) != 9:
            raise RuntimeError("Expected torchvision EfficientNet-B0 features[0:9]")
        for parameter in model.backbone.parameters():
            parameter.requires_grad_(False)
        for index in (7, 8):
            for parameter in features[index].parameters():
                parameter.requires_grad_(True)
        if not all(p.requires_grad for p in model.crop_head.parameters()):
            raise RuntimeError("Crop head was unexpectedly frozen")
        if not all(p.requires_grad for p in model.disease_head.parameters()):
            raise RuntimeError("Disease head was unexpectedly frozen")
        n_frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        if not n_frozen or not n_trainable:
            raise RuntimeError("FT03 must have both frozen and trainable parameters")
        print(f"FT03 Partial Fine-Tuning: frozen features[0:7], trainable features[7:9] "
              f"+ crop/disease heads | frozen={n_frozen:,}, trainable={n_trainable:,}")
        return {"mode": mode, "frozen_features": list(range(7)),
                "trainable_features": [7, 8], "frozen_parameters": n_frozen,
                "trainable_parameters": n_trainable,
                "frozen_bn_eval": True}


    def stage2_reset_classifier_heads(model):
        """FT04 ONLY: reinitialize the two final Linear layers, not the shared backbone.

        Call immediately after restoring Exp04 and seeding, before the optimizer.
        NEVER call when continuing from a trained FT04 checkpoint for final fitting.
        """
        for name in ("crop_head", "disease_head"):
            head = getattr(model, name)
            linear_layers = [module for module in head.modules() if isinstance(module, nn.Linear)]
            if len(linear_layers) != 1:
                raise RuntimeError(f"Expected exactly one nn.Linear in {name}; found {len(linear_layers)}")
            linear_layers[0].reset_parameters()
        print("FT04: reinitialized Crop and Disease Linear heads; retained Exp04 backbone weights")


    def stage2_optimizer_groups(model, backbone_lr, head_lr):
        """Exclude frozen tensors from AdamW, preserving backbone/head LR groups."""
        groups = model.parameter_groups(backbone_lr, head_lr)
        clean = []
        for group in groups:
            params = [p for p in group["params"] if p.requires_grad]
            if params:
                clean.append({**group, "params": params})
        if not clean:
            raise RuntimeError("No trainable optimizer parameters")
        included = sum(p.numel() for g in clean for p in g["params"])
        expected = sum(p.numel() for p in model.parameters() if p.requires_grad)
        if included != expected:
            raise RuntimeError("Optimizer groups do not cover all trainable tensors exactly")
        return clean


    def stage2_train_epoch(model, loader, optimizer, scaler, device, *, amp, crop_weight,
                           disease_weight, freeze_early_features=False, freeze_backbone=False):
        """Weighted CE for fine-tuning; raw crop/disease CE are separately logged."""
        if crop_weight <= 0 or disease_weight <= 0:
            raise ValueError("Loss weights must be positive")
        model.train()
        if freeze_backbone:
            # Essential: requires_grad=False alone does NOT freeze BatchNorm running stats.
            model.backbone.eval()
        elif freeze_early_features:
            # model.train() would otherwise resume BatchNorm running-stat updates and
            # stochastic-depth behavior in frozen blocks. Disable both for features[0:7].
            if not isinstance(model.backbone[0], nn.Sequential) or len(model.backbone[0]) != 9:
                raise RuntimeError("FT03 feature layout changed")
            for block in list(model.backbone[0].children())[:7]:
                block.eval()
        sums = np.zeros(3, dtype=np.float64)
        n_images = correct_crop = correct_disease = correct_pair = 0
        criterion = nn.CrossEntropyLoss()
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            crop = batch["crop"].to(device, non_blocking=True)
            disease = batch["disease"].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            try:
                with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
                    logits = model(images)
                    crop_loss = criterion(logits["crop_logits"], crop)
                    disease_loss = criterion(logits["disease_logits"], disease)
                    weighted_loss = crop_weight * crop_loss + disease_weight * disease_loss
                if not bool(torch.isfinite(weighted_loss)):
                    raise RuntimeError("Nonfinite fine-tuning loss")
                scaler.scale(weighted_loss).backward()
                scaler.step(optimizer)
                scaler.update()
            except torch.cuda.OutOfMemoryError as err:
                raise RuntimeError("CUDA OOM: stop, document batch size change, and rerun with a NEW ID") from err
            count = images.size(0)
            pred_crop = logits["crop_logits"].argmax(1)
            pred_disease = logits["disease_logits"].argmax(1)
            correct_crop += int((pred_crop == crop).sum().item())
            correct_disease += int((pred_disease == disease).sum().item())
            correct_pair += int(((pred_crop == crop) & (pred_disease == disease)).sum().item())
            sums += count * np.array([weighted_loss.item(), crop_loss.item(), disease_loss.item()])
            n_images += count
        if n_images == 0:
            raise ValueError("Empty fine-tuning DataLoader")
        losses = sums / n_images
        return {"total_loss": float(losses[0]), "crop_loss": float(losses[1]),
                "disease_loss": float(losses[2]), "crop_top1": correct_crop / n_images,
                "disease_top1": correct_disease / n_images,
                "composition_top1": correct_pair / n_images}


### 10A. Development fine-tuning functions and controls (OFF)

FT01–FT04 were development experiments; the code covers alternative fine-tuning / freezing / head-retraining strategies. The original Exp04 checkpoint remains the selected source for the final fit. **Keep `RUN_STAGE2_DEVELOPMENT=False`.** Refer to individual run configurations and metrics in the original `outputs/stage2_development/` folders before making specific claims about their individual outcomes.


In [ ]:
if RUN_RECORDED_WORKFLOW:
    def run_stage2_development(parent_checkpoint, *, ft_id, max_epochs=3,
                               backbone_lr=1e-6, head_lr=1e-4, requested_workers=4,
                               approved=False):
        if not approved:
            raise PermissionError("Explicit approval required before Stage 2A development fine-tuning")
        if ft_id not in STAGE2_DEVELOPMENT_PRESETS:
            raise ValueError("Choose ft01, ft02, ft03, or ft04; do not overwrite old experiment IDs")
        if not isinstance(max_epochs, int) or max_epochs < 1:
            raise ValueError("max_epochs must be a positive integer")
        path, ck, split_frames, report, candidates = stage2_read_parent(parent_checkpoint, (1,))
        if ck["config"].get("experiment_id") != "exp04":
            raise ValueError("All FT runs must start independently from Exp04")
        if not (0 < backbone_lr < float(ck["config"]["backbone_lr"])):
            raise ValueError("Stage 2A backbone LR must be smaller than Exp04 LR")
        if ft_id == "ft04":
            # Freshly reset classification heads need the normal Exp04 head LR.
            if not (0 < head_lr <= float(ck["config"]["head_lr"])):
                raise ValueError("FT04 head LR must be positive and at most Exp04 head LR")
        elif not (0 < head_lr < float(ck["config"]["head_lr"])):
            raise ValueError("FT01–FT03 head LR must be smaller than Exp04 head LR")
        dest = OUTPUT_ROOT / "stage2_development" / ft_id
        if dest.exists():
            raise FileExistsError(f"{dest} already exists; do not overwrite saved results")
        cfg = copy.deepcopy(ck["config"])
        cfg.update({"experiment_id": ft_id, "stage2_mode": "development",
                    "parent_checkpoint": str(path), "backbone_lr": float(backbone_lr),
                    "head_lr": float(head_lr), "max_epochs": max_epochs,
                    "scheduler": "constant low LR", **STAGE2_DEVELOPMENT_PRESETS[ft_id],
                    "requested_num_workers": int(requested_workers)})
        if ft_id == "ft03":
            cfg["changed_parameter"] = (
                "Partial Fine-Tuning vs FT01: freeze EfficientNet-B0 features[0:7] "
                "(including BN running statistics); train features[7:9] and both heads"
            )
        if ft_id == "ft04":
            cfg["changed_parameter"] = (
                "cRT-inspired head retraining: keep Exp04 backbone frozen including BN statistics; "
                "reset both Linear heads and train with tempered disease-balanced sampling"
            )
            cfg["head_reinitialized"] = True
            cfg["head_reinitialization_seed"] = int(cfg["seed"])
            cfg["head_reinitialization_source"] = "fresh Linear.reset_parameters after Exp04 loading"
            cfg["early_stopping_patience"] = 5
        seed_everything(cfg["seed"], cfg)
        datasets = {k: DPDDataset(frame, IMAGE_ROOT, training_transform(cfg) if k == "train_fit"
                                  else validation_transform()) for k, frame in split_frames.items()}
        cfg["num_workers"] = stage2_choose_workers(datasets, cfg, requested_workers)
        train_sampler = stage2_build_sampler(split_frames["train_fit"], cfg)
        loaders = {k: make_loader(ds, cfg, shuffle=(k == "train_fit" and train_sampler is None),
                                  sampler=train_sampler if k == "train_fit" else None)
                   for k, ds in datasets.items()}
        model = stage2_restore_model(ck, DEVICE)
        if cfg["fine_tune_mode"] == "head_only_retrain":
            stage2_reset_classifier_heads(model)
        trainability = stage2_configure_trainability(model, cfg["fine_tune_mode"])
        cfg["trainability"] = trainability
        optimizer = torch.optim.AdamW(stage2_optimizer_groups(model, backbone_lr, head_lr),
                                      weight_decay=float(cfg["weight_decay"]))
        use_amp = bool(cfg["amp"] and DEVICE.type == "cuda")
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        dest.mkdir(parents=True, exist_ok=False)
        parent_hash = sha256_file(path)
        (dest / "config.json").write_text(json.dumps({"config": cfg, "parent_checkpoint_sha256": parent_hash,
            "development_split": report, "candidate_policy_status": "locked to Exp04 pairs",
            "environment": environment_details()}, indent=2), encoding="utf-8")
        history = []
        best_hm = -math.inf
        best_epoch = None
        start = time.perf_counter()
        bad_epochs = 0
        for epoch in range(1, max_epochs + 1):
            epoch_start = time.perf_counter()
            train = stage2_train_epoch(model, loaders["train_fit"], optimizer, scaler, DEVICE,
                                       amp=use_amp, crop_weight=cfg["crop_loss_weight"],
                                       disease_weight=cfg["disease_loss_weight"],
                                       freeze_early_features=(cfg["fine_tune_mode"] == "partial_last2"),
                                       freeze_backbone=(cfg["fine_tune_mode"] == "head_only_retrain"))
            seen, _ = evaluate_loader(model, loaders["val_seen"], DEVICE, candidates, use_amp)
            unseen, _ = evaluate_loader(model, loaders["val_unseen"], DEVICE, candidates, use_amp)
            joint = generalisation_metrics(seen, unseen)
            row = {"epoch": epoch, "backbone_lr": backbone_lr, "head_lr": head_lr,
                   **{f"train_{k}": v for k, v in train.items()},
                   **{f"seen_{k}": v for k, v in seen.items()},
                   **{f"unseen_{k}": v for k, v in unseen.items()},
                   **joint, "epoch_seconds": time.perf_counter() - epoch_start}
            history.append(row)
            pd.DataFrame(history).to_csv(dest / "training_history.csv", index=False)
            if joint["composition_harmonic_mean"] > best_hm + 1e-6:
                bad_epochs = 0
                best_epoch = epoch
                best_hm = joint["composition_harmonic_mean"]
                best_metrics = {"epoch": epoch, "seen": seen, "unseen": unseen, **joint}
                atomic_checkpoint({"stage": "stage2_development", "epoch": epoch,
                    "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                    "scaler_state": scaler.state_dict(), "config": cfg,
                    "candidate_pairs": ck["candidate_pairs"],
                    "class_mappings": ck["class_mappings"],
                    "composition_mapping": ck["composition_mapping"],
                    "development_split": report, "parent_checkpoint": str(path),
                    "parent_checkpoint_sha256": parent_hash,
                    "parent_best_epoch": ck["epoch"], "best_metrics": best_metrics,
                    "environment": environment_details()}, dest / "best_development_finetuned.pt")
            else:
                bad_epochs += 1
            print(f"{ft_id.upper()} epoch {epoch:02d}/{max_epochs} | "
                  f"weighted train loss {train['total_loss']:.4f} | seen {seen['composition_top1']:.2%} | "
                  f"unseen {unseen['composition_top1']:.2%} | HM {joint['composition_harmonic_mean']:.2%} "
                  f"| {row['epoch_seconds']:.1f}s" + (" * best" if best_epoch == epoch else ""))
            if ft_id == "ft04" and bad_epochs >= int(cfg["early_stopping_patience"]):
                print(f"FT04 early stopping after {bad_epochs} non-improving epochs")
                break
        if best_epoch is None:
            raise RuntimeError("Stage 2A did not produce a valid checkpoint")
        result = {**best_metrics, "epochs_run": len(history),
                  "training_seconds": time.perf_counter() - start,
                  "accuracy_units": "fractions in [0,1]"}
        (dest / "validation_metrics.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
        pd.DataFrame([{ "experiment_id": ft_id, "parent": str(path),
            "best_epoch": best_epoch, "seen_composition_top1": best_metrics["seen"]["composition_top1"],
            "unseen_composition_top1": best_metrics["unseen"]["composition_top1"],
            "composition_harmonic_mean": best_hm, "crop_loss_weight": cfg["crop_loss_weight"],
            "disease_loss_weight": cfg["disease_loss_weight"],
            "checkpoint_path": str(dest / "best_development_finetuned.pt")}
            ]).to_csv(dest / "experiment_summary.csv", index=False)
        fig = plot_training_history(history, best_epoch, dest / "training_curves.png")
        plt.show()
        plt.close(fig)
        fig = plot_task_accuracy(history, best_epoch, dest / "task_accuracy_curves.png")
        plt.close(fig)
        print(f"Stage 2A complete: best HM {best_hm:.2%} at additional epoch {best_epoch}. "
              f"Exp04 reference HM = {ck['best_metrics']['composition_harmonic_mean']:.2%}.")
        print("Saved development-only results to:", dest)
        return result, dest


In [ ]:
if RUN_RECORDED_WORKFLOW:
    # 10A CONTROLS — only change these when you EXPLICITLY approve a development fine-tuning run.
    RUN_STAGE2_DEVELOPMENT = False
    FT_EXPERIMENT_ID = "ft04"  # cRT-inspired head-only retraining; starts independently from Exp04.
    FT_SOURCE_CHECKPOINT = OUTPUT_ROOT / "exp04" / "best_development.pt"
    FT_MAX_EPOCHS = 12   # Fresh heads need more than 3 epochs; FT04 has patience=5
    FT_BACKBONE_LR = 1e-6
    FT_HEAD_LR = 1e-3     # Heads are reset from scratch; frozen backbone LR is unused
    FT_REQUESTED_WORKERS = 4   # Preflight: try 4, then 2, then 0 if needed.

    if RUN_STAGE2_DEVELOPMENT:
        stage2_dev_metrics, stage2_dev_dir = run_stage2_development(
            FT_SOURCE_CHECKPOINT, ft_id=FT_EXPERIMENT_ID, max_epochs=FT_MAX_EPOCHS,
            backbone_lr=FT_BACKBONE_LR, head_lr=FT_HEAD_LR,
            requested_workers=FT_REQUESTED_WORKERS, approved=True)
    else:
        print("Stage 2A OFF. No development fine-tuning was run.")


### 10B. Final full-data fine-tuning functions and controls (OFF)

The final checkpoint is **already trained**. Its parent is `outputs/exp04/best_development.pt`; 8,000 training images, three additional epochs, and low learning rates were fixed before official testing. Output folders refuse overwrites. **Keep both `RUN_STAGE2_FINAL=False` and `APPROVE_FULL_DATA_TRAINING=False`.**


In [ ]:
if RUN_RECORDED_WORKFLOW:
    def run_stage2_final(parent_checkpoint, *, final_run_id, additional_epochs=3,
                         requested_workers=4, approved=False,
                         fallback_backbone_lr=1e-6, fallback_head_lr=1e-4,
                         fallback_crop_weight=1.0, fallback_disease_weight=1.0):
        if not approved:
            raise PermissionError("Explicit approval required before Stage 2B final full-data training")
        if not isinstance(additional_epochs, int) or additional_epochs < 1:
            raise ValueError("additional_epochs must be a positive integer")
        if not final_run_id or Path(final_run_id).name != final_run_id or final_run_id in (".", ".."):
            raise ValueError("Use a simple unique final_run_id without path separators")
        path, ck, split_frames, report, _ = stage2_read_parent(
            parent_checkpoint, allowed_stages=(1, "stage2_development"))
        if ck["stage"] == 1 and ck["config"].get("experiment_id") != "exp04":
            raise ValueError("Final Stage 1 fallback must be Exp04")
        dest = OUTPUT_ROOT / "stage2_final" / final_run_id
        if dest.exists():
            raise FileExistsError(f"{dest} already exists; use a NEW final run ID")
        # All 8,000 are from the original TRAIN CSV, NOT the official test sets.
        full_frame = pd.concat([split_frames[k] for k in ("train_fit", "val_seen", "val_unseen")],
                               ignore_index=True)
        if len(full_frame) != 8000 or full_frame["filename"].nunique() != 8000:
            raise ValueError("Full-data training requires 8,000 distinct original training images")
        cfg = copy.deepcopy(ck["config"])
        if ck["stage"] == 1:
            cfg.update({"backbone_lr": float(fallback_backbone_lr),
                        "head_lr": float(fallback_head_lr),
                        "crop_loss_weight": float(fallback_crop_weight),
                        "disease_loss_weight": float(fallback_disease_weight)})
        if cfg["backbone_lr"] <= 0 or cfg["head_lr"] <= 0:
            raise ValueError("Final training LRs must be positive")
        if cfg["crop_loss_weight"] <= 0 or cfg["disease_loss_weight"] <= 0:
            raise ValueError("Final loss weights must be positive")
        cfg.update({"experiment_id": final_run_id, "stage2_mode": "final_full_data",
                    "additional_epochs": additional_epochs, "scheduler": "constant low LR",
                    "requested_num_workers": int(requested_workers),
                    "parent_checkpoint": str(path)})
        seed_everything(cfg["seed"], cfg)
        dataset = DPDDataset(full_frame, IMAGE_ROOT, training_transform(cfg))
        # Reuse the Section 4 Windows worker preflight using a single full-data dataset.
        probes = {"train_fit": dataset, "val_seen": dataset, "val_unseen": dataset}
        cfg["num_workers"] = stage2_choose_workers(probes, cfg, requested_workers)
        sampler = stage2_build_sampler(full_frame, cfg)
        loader = make_loader(dataset, cfg, shuffle=(sampler is None), sampler=sampler)
        model = stage2_restore_model(ck, DEVICE)
        # Preserve FT03 or FT04 freeze strategy if its checkpoint wins development.
        # Never reset FT04's classifier weights here: they are already trained.
        cfg["fine_tune_mode"] = cfg.get("fine_tune_mode", "full")
        cfg["trainability"] = stage2_configure_trainability(model, cfg["fine_tune_mode"])
        optimizer = torch.optim.AdamW(stage2_optimizer_groups(model, cfg["backbone_lr"], cfg["head_lr"]),
                                      weight_decay=float(cfg["weight_decay"]))
        use_amp = bool(cfg["amp"] and DEVICE.type == "cuda")
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        dest.mkdir(parents=True, exist_ok=False)
        parent_hash = sha256_file(path)
        (dest / "config.json").write_text(json.dumps({"config": cfg,
            "parent_checkpoint_sha256": parent_hash, "original_training_count": 8000,
            "former_validation_is_independent": False, "development_split": report,
            "candidate_policy_status": "locked to parent checkpoint pairs",
            "environment": environment_details()}, indent=2), encoding="utf-8")
        history = []
        started = time.perf_counter()
        for epoch in range(1, additional_epochs + 1):
            start_epoch = time.perf_counter()
            training = stage2_train_epoch(model, loader, optimizer, scaler, DEVICE,
                amp=use_amp, crop_weight=cfg["crop_loss_weight"],
                disease_weight=cfg["disease_loss_weight"],
                freeze_early_features=(cfg["fine_tune_mode"] == "partial_last2"),
                freeze_backbone=(cfg["fine_tune_mode"] == "head_only_retrain"))
            row = {"additional_epoch": epoch, "backbone_lr": cfg["backbone_lr"],
                   "head_lr": cfg["head_lr"], "crop_loss_weight": cfg["crop_loss_weight"],
                   "disease_loss_weight": cfg["disease_loss_weight"],
                   **training, "epoch_seconds": time.perf_counter() - start_epoch}
            history.append(row)
            pd.DataFrame(history).to_csv(dest / "full_data_history.csv", index=False)
            print(f"FINAL epoch {epoch:02d}/{additional_epochs} | "
                  f"weighted train loss {training['total_loss']:.4f} | {row['epoch_seconds']:.1f}s")
        atomic_checkpoint({"stage": 2, "epoch": additional_epochs,
            "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
            "scaler_state": scaler.state_dict(), "config": cfg,
            "candidate_pairs": ck["candidate_pairs"], "class_mappings": ck["class_mappings"],
            "composition_mapping": ck["composition_mapping"], "development_split": report,
            "parent_checkpoint": str(path), "parent_checkpoint_sha256": parent_hash,
            "parent_stage": ck["stage"], "parent_epoch": ck["epoch"],
            "full_training_count": len(full_frame),
            "former_validation_is_independent": False,
            "training_seconds": time.perf_counter() - started,
            "environment": environment_details()}, dest / "final_full_data.pt")
        model.eval()
        print("Final full-data model saved to:", dest)
        print("NO development validation or official tests were run in Stage 2B.")
        return dest


In [ ]:
if RUN_RECORDED_WORKFLOW:
    # 10B CONTROLS — KEEP FALSE until you reviewed all development experiments and explicitly APPROVE final fit.
    RUN_STAGE2_FINAL = False
    APPROVE_FULL_DATA_TRAINING = False  # Separate approval guard.
    FINAL_RUN_ID = "b2_final_exp04"  # Historical completed run ID; do not rerun.
    FINAL_SOURCE_CHECKPOINT = (OUTPUT_ROOT / "exp04" / "best_development.pt")
    FINAL_ADDITIONAL_EPOCHS = 3   # Predeclared; do not select using former validation after full fit.
    FINAL_REQUESTED_WORKERS = 4

    if RUN_STAGE2_FINAL:
        if not APPROVE_FULL_DATA_TRAINING:
            raise PermissionError("Full-data training not approved. Keep both controls False until ready.")
        stage2_final_directory = run_stage2_final(
            FINAL_SOURCE_CHECKPOINT, final_run_id=FINAL_RUN_ID,
            additional_epochs=FINAL_ADDITIONAL_EPOCHS,
            requested_workers=FINAL_REQUESTED_WORKERS, approved=True)
    else:
        print("Stage 2B OFF. No 8,000-image training or official tests were run.")


## 11. Official test results (completed)

The **same frozen Stage 2B checkpoint** was evaluated once on **2,000 official seen** and **1,448 official unseen** images with 9,625 candidates. Scores below are **composition Top-1**, with harmonic mean across seen/unseen accuracies.

| Method | Seen Top-1 | Unseen Top-1 | Harmonic mean |
|:--|--:|--:|--:|
| **Original B2 (primary, γ=0)** | **55.60%** | **1.86%** | **3.61%** |
| B2 + preselected calibration (γ=4) | 38.70% | 4.77% | 8.49% |

**Detailed original B2 metrics:** Seen crop Top-1 **84.65%**, disease Top-1 **59.65%**; unseen crop Top-1 **61.60%**, disease Top-1 **7.94%**. Calibration only changes **composition ranking**; separate crop/disease head predictions remain unchanged.

**Interpretation:** Generalisation to unseen crop–disease combinations is weak, particularly the disease head. Calibration increases unseen accuracy and HM at the cost of seen accuracy. This trade-off is an optional additional experiment, **not evidence that the B2 architecture itself changed**. Report both modes distinctly; do not retune gamma from official results.

The [official comparison](../../reports/official/comparison.json) and the per-mode metrics and confusion plots are available under reports/official/. Full prediction and error files remain with the original local evaluation outputs.

### 11.1. Limitations and reproducibility

Unseen disease Top-1 was 7.94%, compared with 61.60% for the crop head. The low unseen disease accuracy constrains composition accuracy. Penalising seen pairs shifts errors between the seen and unseen partitions. The final calibration mask contains 323 training-seen pairs derived from all 8,000 training images. The development mask contains 313 pairs.

The teammate-reported B1 harmonic mean was 4.38%. The B1 source metrics were unavailable for verification. Differences in preprocessing, training and model selection limit attribution to the architecture.

The [saved configuration](../../reports/final_training/config.json) records the environment, parent checkpoint and split hashes. The [handover](../../docs/HANDOVER.md) identifies the original local artifacts required for reproduction.

## 12. Archived official evaluation workflow

The frozen checkpoint was evaluated with Original B2 scoring (γ = 0) and the preselected calibration penalty (γ = 4) using one shared inference pass and a single view per image. [Saved results](../../reports/official/comparison.json) are available for both modes. The local checkpoint and calibration lock are needed only for the optional preflight or the guarded historical evaluation workflow. Both controls below default to False.

In [ ]:
if RUN_RECORDED_WORKFLOW:
    # Archived final evaluation workflow — official test already completed. OFF by default.
    import subprocess

    FINAL_DIR = PROJECT_DIR / "outputs" / "stage2_final" / "b2_final_exp04"
    FINAL_CHECKPOINT = FINAL_DIR / "final_full_data.pt"
    FINAL_LOCK = FINAL_DIR / "final_eval_lock.json"  # Preselected gamma=4 from Exp06; do not retune
    TRAINING_SPLITS = PROJECT_DIR / "splits_v2"  # Original TRAINING splits only
    EVAL_SCRIPT = PROJECT_DIR / "evaluate.py"

    RUN_FINAL_PREFLIGHT = False
    RUN_OFFICIAL_TEST = False  # Keep False until all final settings/paths approved.
    # Official data paths are intentionally excluded from the shared notebook:
    OFFICIAL_SEEN_CSV = None        # e.g. Path(r"C:\path\official_seen.csv")
    OFFICIAL_UNSEEN_CSV = None      # e.g. Path(r"C:\path\official_unseen.csv")
    OFFICIAL_SEEN_IMAGE_ROOT = None
    OFFICIAL_UNSEEN_IMAGE_ROOT = None
    OFFICIAL_OUTPUT = FINAL_DIR / "official_evaluation_both"  # Must NOT exist before first official run

    preflight_cmd = [sys.executable, str(EVAL_SCRIPT), "--preflight",
                     "--checkpoint", str(FINAL_CHECKPOINT),
                     "--lock-json", str(FINAL_LOCK),
                     "--training-splits", str(TRAINING_SPLITS)]
    if RUN_FINAL_PREFLIGHT:
        subprocess.run(preflight_cmd, check=True)

    if RUN_OFFICIAL_TEST:
        test_paths = [OFFICIAL_SEEN_CSV, OFFICIAL_UNSEEN_CSV,
                      OFFICIAL_SEEN_IMAGE_ROOT, OFFICIAL_UNSEEN_IMAGE_ROOT]
        if any(v is None for v in test_paths):
            raise ValueError("Fill all four official test paths before running the one-time evaluation")
        if OFFICIAL_OUTPUT.exists():
            raise FileExistsError(f"Official output already exists; refusing to overwrite: {OFFICIAL_OUTPUT}")
        cmd = [sys.executable, str(EVAL_SCRIPT),
               "--official", "--approve-official-testing", "--mode", "both",
               "--checkpoint", str(FINAL_CHECKPOINT),
               "--lock-json", str(FINAL_LOCK),
               "--training-splits", str(TRAINING_SPLITS),
               "--seen-csv", str(OFFICIAL_SEEN_CSV),
               "--unseen-csv", str(OFFICIAL_UNSEEN_CSV),
               "--seen-image-root", str(OFFICIAL_SEEN_IMAGE_ROOT),
               "--unseen-image-root", str(OFFICIAL_UNSEEN_IMAGE_ROOT),
               "--output", str(OFFICIAL_OUTPUT)]
        subprocess.run(cmd, check=True)
        print("Official metrics saved separately to:", OFFICIAL_OUTPUT)
    else:
        print("Official evaluation is disabled.")


## Saved evidence and interpretation

This section was assembled from existing local artifacts on 23 November 2026. It does not rerun any model. Historical cell outputs below/above remain unchanged.

### Setup and recorded hyperparameters

| Run | Seed | Batch | LR backbone / head | WD | Crop / disease weight | Sampling |
| --- | --- | --- | --- | --- | --- | --- |
| exp01 | 42 | 32 | 1e-05 / 0.001 | 0.0001 | 1.0 / 1.0 | natural shuffle |
| exp02 | 42 | 32 | 1e-05 / 0.001 | 0.0001 | 1.0 / 1.0 | natural shuffle |
| exp03 | 42 | 32 | 1e-05 / 0.0005 | 0.0001 | 1.0 / 1.0 | natural shuffle |
| exp04 | 42 | 32 | 1e-05 / 0.001 | 0.0001 | 1.0 / 1.0 | disease_tempered |
| exp05 | 42 | 32 | 1e-05 / 0.001 | 0.0001 | 1.0 / 1.0 | disease_tempered |

Exact augmentation, scheduler, class mappings and environment versions are retained in the linked configuration files. Current source settings need not reproduce a historical execution’s environment.

### Dataset and method

Development uses train-fit 6,905 / seen validation 695 / unseen validation 400, with original class IDs and all 9,625 candidate pairs. Validation is deterministic BICUBIC Resize(256), CenterCrop(224), ImageNet normalisation unless a specifically labelled resolution ablation is discussed. Training augmentation and sampling vary by experiment; refer to each saved configuration. No official test metric is used here for model selection.

### Development validation results

| Run | Best epoch | Task | Top-k | Seen (%) | Unseen (%) | HM (%) |
| --- | --- | --- | --- | --- | --- | --- |
| exp01 | 11 | crop | 1 | 86.1871 | 66.5000 | 75.0743 |
| exp01 | 11 | crop | 5 | 95.5396 | 77.0000 | 85.2737 |
| exp01 | 11 | disease | 1 | 70.6475 | 17.5000 | 28.0514 |
| exp01 | 11 | disease | 5 | 88.3453 | 31.5000 | 46.4412 |
| exp01 | 11 | composition | 1 | 66.1871 | 2.0000 | 3.8827 |
| exp01 | 11 | composition | 5 | 83.4532 | 9.2500 | 16.6541 |
| exp02 | 13 | crop | 1 | 85.3237 | 64.2500 | 73.3023 |
| exp02 | 13 | crop | 5 | 95.3957 | 76.5000 | 84.9093 |
| exp02 | 13 | disease | 1 | 71.2230 | 19.5000 | 30.6173 |
| exp02 | 13 | disease | 5 | 88.3453 | 31.7500 | 46.7123 |
| exp02 | 13 | composition | 1 | 66.4748 | 1.7500 | 3.4102 |
| exp02 | 13 | composition | 5 | 83.7410 | 9.2500 | 16.6598 |
| exp03 | 14 | crop | 1 | 85.7554 | 63.7500 | 73.1332 |
| exp03 | 14 | crop | 5 | 95.8273 | 76.2500 | 84.9250 |
| exp03 | 14 | disease | 1 | 70.7914 | 19.0000 | 29.9591 |
| exp03 | 14 | disease | 5 | 89.0647 | 30.2500 | 45.1614 |
| exp03 | 14 | composition | 1 | 66.4748 | 1.2500 | 2.4539 |
| exp03 | 14 | composition | 5 | 84.0288 | 7.7500 | 14.1911 |
| exp04 | 13 | crop | 1 | 86.0432 | 60.2500 | 70.8728 |
| exp04 | 13 | crop | 5 | 95.8273 | 76.7500 | 85.2342 |
| exp04 | 13 | disease | 1 | 70.6475 | 19.5000 | 30.5638 |
| exp04 | 13 | disease | 5 | 88.6331 | 32.7500 | 47.8276 |
| exp04 | 13 | composition | 1 | 66.4748 | 2.2500 | 4.3527 |
| exp04 | 13 | composition | 5 | 84.0288 | 10.0000 | 17.8730 |
| exp05 | 13 | crop | 1 | 85.7554 | 60.2500 | 70.7750 |
| exp05 | 13 | crop | 5 | 95.9712 | 76.7500 | 85.2911 |
| exp05 | 13 | disease | 1 | 69.2086 | 18.7500 | 29.5062 |
| exp05 | 13 | disease | 5 | 88.4892 | 32.2500 | 47.2718 |
| exp05 | 13 | composition | 1 | 64.7482 | 1.7500 | 3.4079 |
| exp05 | 13 | composition | 5 | 84.0288 | 9.7500 | 17.4726 |

Values come from each run’s `validation_metrics.json`. Top-1 composition HM is the saved checkpoint selection criterion. Other HMs above are descriptive calculations from the two saved accuracies, not additional experiments. Validation `total_loss` in the JSON is unweighted crop CE + disease CE; weighted training totals require task-wise weighting before comparison.

### Visualisation and artifact sources


**exp01**

[config.json](../../outputs/exp01/config.json), [validation_metrics.json](../../outputs/exp01/validation_metrics.json), [training_history.csv](../../outputs/exp01/training_history.csv)


![exp01 — training_curves.png](../../outputs/exp01/training_curves.png)


**exp02**

[config.json](../../outputs/exp02/config.json), [validation_metrics.json](../../outputs/exp02/validation_metrics.json), [training_history.csv](../../outputs/exp02/training_history.csv)


![exp02 — training_curves.png](../../outputs/exp02/training_curves.png)


**exp03**

[config.json](../../outputs/exp03/config.json), [validation_metrics.json](../../outputs/exp03/validation_metrics.json), [training_history.csv](../../outputs/exp03/training_history.csv)


![exp03 — training_curves.png](../../outputs/exp03/training_curves.png)


**exp04**

[config.json](../../outputs/exp04/config.json), [validation_metrics.json](../../outputs/exp04/validation_metrics.json), [training_history.csv](../../outputs/exp04/training_history.csv)


![exp04 — training_curves.png](../../outputs/exp04/training_curves.png)


**exp05**

[config.json](../../outputs/exp05/config.json), [validation_metrics.json](../../outputs/exp05/validation_metrics.json), [training_history.csv](../../outputs/exp05/training_history.csv)


![exp05 — training_curves.png](../../outputs/exp05/training_curves.png)

### Discussion and limitations

Exp04 is the selected historical EfficientNet recipe, with development composition HM 4.35%. These backbone variants belong to Baseline 2 and both use a shared feature extractor with separate crop and disease heads. This clean overview has no original executed cells; the historical notebooks and saved artifacts supply the evidence. Do not treat this overview as a newly executed experiment.


The original executed training history remains in [B2_HISTORY_v1_initial_training.ipynb](../../archive/B2_HISTORY_v1_initial_training.ipynb) and [B2_HISTORY_v2_training_and_diagnostics.ipynb](../../archive/B2_HISTORY_v2_training_and_diagnostics.ipynb). No executed output has been manufactured for this clean overview.
